# Introduction: Why Calculus for Machine Learning

## What's covered

- Why calculus is the second math foundation for ML (after linear algebra) — the **four jobs** calculus does in every ML system
- The roadmap through this repo — 8 notebooks from limits to expectation
- **Limits** — what `lim_{x \to a} f(x)` actually means, with numerical intuition
- **Continuity** — and why ReLU's kink at zero is calculus's most-studied corner in modern ML
- **Limits at infinity** — saturation behavior of sigmoid, vanishing gradients
- The slope-of-a-curve setup — a preview of the **derivative**, which becomes the next notebook


## Why calculus for machine learning

Linear algebra gave us vectors, matrices, and the linear models built from them. Calculus is the toolkit for everything that **changes** — and learning is *all* about change. Every weight update in every model is a calculus question.

Calculus does four jobs in ML, and the eight notebooks of this repo each serve at least one of them:

**1. Find minima of loss functions.** Every training procedure is the same procedure: write down a scalar loss, compute its gradient with respect to the parameters, take a step downhill. Repeat. The whole machinery of gradient descent, momentum, Adam, learning-rate schedules, and second-order methods is calculus — *how* to use derivative information to find good parameters.

**2. Measure sensitivity.** "If I nudge this input by `dx`, how much does the output change?" That's the **derivative**. Generalized to many inputs and many outputs, it's the **Jacobian**. In ML this question shows up as: feature importance, adversarial robustness, influence functions, model interpretation, integrated gradients.

**3. Approximate complex things with simple ones.** A neural network is a complicated function. Near any single point, calculus lets us approximate it with a *linear* function (the first-order Taylor expansion) or a *quadratic* function (second-order). Newton's method, Laplace approximation, variational inference, and most theoretical results about training dynamics rely on this trick.

**4. Aggregate over continuous distributions.** "What's the average loss across all possible inputs?" That's an **integral**. Expectation, KL divergence, evidence lower bounds, change-of-variables for normalizing flows — all integrals in disguise.

Memorize those four jobs. They are the entire reason calculus shows up in machine learning.


## The roadmap

We start with the foundations and build out toward the operations a real training loop uses every microsecond.

1. **This notebook** — limits and continuity, the language calculus is written in.
2. **Derivatives** — single-variable derivatives, the rules (product, quotient, chain), and the derivatives of every activation function you've heard of.
3. **Partial derivatives and gradients** — derivatives in many dimensions; the gradient as the direction of steepest ascent.
4. **The chain rule and backpropagation** — the chain rule scaled up from scalars to vectors to matrices, then used to derive backprop on a tiny network end-to-end.
5. **Jacobians, Hessians, and matrix calculus** — vector-valued derivatives, second derivatives, and the identities you'll write on whiteboards in interviews.
6. **Taylor series and approximations** — linear and quadratic local approximations; the math behind Newton's method and variational inference.
7. **Optimization and convexity** — gradient descent variants, convexity, second-order conditions, saddle points.
8. **Integration and expectation** — the bridge into probability and statistics; the foundation for KL divergence and normalizing flows.

By the end you'll be able to derive backpropagation from scratch, explain why Adam differs from SGD, and read the gradient-flow equations behind modern training papers.


## What is a limit?

A limit answers the question: *what value does `f(x)` get arbitrarily close to as `x` gets arbitrarily close to `a` — without `x` necessarily ever reaching `a`?*

We write this as:

$$
\lim_{x \to a} f(x) = L
$$

The point of the limit notation is that we **never substitute `x = a` directly**. We only care about behavior in the *neighborhood* of `a`. That distinction matters in three settings:

- `f` is **undefined** at `a` — for example, `f(x) = sin(x)/x` at `x = 0` is the form `0/0`, but `lim_{x→0} sin(x)/x = 1`. The function has a "removable hole" that the limit fills in.
- `f` is **defined** at `a` but the limit disagrees — this is a discontinuity.
- `f` **explodes** near `a` — the limit is `±∞` (or doesn't exist).

There is a formal `ε-δ` definition that says "for every desired closeness `ε > 0` of `f(x)` to `L`, there's a tightness `δ > 0` such that `|x - a| < δ` forces `|f(x) - L| < ε`." We won't push on the formalism — for ML, the *intuition* of "you can get as close as you want" is what carries through.

The single most important limit in this entire repo is the one that **defines the derivative**:

$$
f'(x) = \lim_{h \to 0} \frac{f(x + h) - f(x)}{h}
$$

Notebook 2 is the next 14 cells unpacking this. Today we just confirm we can read it.


In [ ]:
import numpy as np

# Numerical demonstration of lim_{x -> 0} sin(x) / x = 1
# We can't substitute x = 0 (gives 0/0), but we can shrink x toward 0 and watch
for x in [1.0, 0.1, 0.01, 0.001, 1e-6, 1e-9]:
    print(f"x = {x:<10g}  sin(x)/x = {np.sin(x) / x:.12f}")

# Numerical demonstration of lim_{h -> 0} (f(x+h) - f(x)) / h = f'(x)
# Take f(x) = x^2, so f'(x) = 2x. At x = 3, we expect 6.
f = lambda x: x ** 2
x = 3.0
print("\nLimit definition of derivative for f(x) = x^2 at x = 3 (expected 6):")
for h in [1.0, 0.1, 0.01, 0.001, 1e-6, 1e-9]:
    slope = (f(x + h) - f(x)) / h
    print(f"h = {h:<10g}  slope = {slope:.10f}")


## Continuity

A function `f` is **continuous at `a`** if

$$
\lim_{x \to a} f(x) = f(a)
$$

In words: the limit exists, the function is defined at `a`, and the two agree. No holes, no jumps, no spikes.

A function is **continuous on an interval** if it is continuous at every point in the interval. Polynomials, exponentials, logarithms, `sin`, `cos`, `tanh`, `sigmoid`, and `softplus` are continuous everywhere they're defined.

**Why ML cares about continuity.** Continuous functions don't have to be differentiable, but discontinuous ones definitely *aren't*. Three concrete examples that show up in modern architectures:

- **The step function** (the old perceptron's output) is discontinuous at 0. Its derivative is zero everywhere it exists and undefined where it jumps — gradient descent has nothing to work with. This is *the* reason historical neural networks couldn't learn.
- **Sigmoid** and **tanh** are continuous and differentiable everywhere — that's what made backprop possible in the first place.
- **ReLU** (`max(0, x)`) is continuous everywhere but **not differentiable at `x = 0`** — it has a corner. In practice, frameworks just pick the subgradient `0` or `1` at zero and move on. It works because we almost never hit exactly `x = 0` numerically. But the corner is real, and it's why "smooth" alternatives like GELU, Swish, and Mish exist — they're ReLU with the kink rounded off.

The distinction between **continuity** (no jumps) and **differentiability** (smooth, has a slope) becomes the central tension of the next notebook.


In [ ]:
# ReLU vs a smooth alternative (softplus) at x = 0
def relu(x): return np.maximum(0.0, x)
def softplus(x): return np.log1p(np.exp(x))   # log(1 + e^x), the smooth ReLU

# Both are continuous at 0
print("Both continuous at 0?")
for x in [-0.001, -1e-6, 0.0, 1e-6, 0.001]:
    print(f"  x = {x:<10g}  relu = {relu(x):.6f}   softplus = {softplus(x):.6f}")

# But the slope behaves very differently near 0
print("\nSlope just to the left vs just to the right of 0:")
h = 1e-6
for name, f in [("relu", relu), ("softplus", softplus)]:
    left  = (f(-h) - f(-2 * h)) / h    # slope just left of 0
    right = (f(2 * h) - f(h)) / h      # slope just right of 0
    print(f"  {name:<8} left slope = {left:.6f},  right slope = {right:.6f}")
print("\nReLU jumps from slope 0 to slope 1 at x = 0 — it's a corner, not smooth.")
print("Softplus glides through 0 — slope passes continuously through 0.5.")


## Limits at infinity — saturation and vanishing gradients

Limits don't have to be at finite points. We can also ask what happens as `x → ∞` or `x → -∞`:

$$
\lim_{x \to \infty} \sigma(x), \qquad \lim_{x \to -\infty} \sigma(x)
$$

For the **sigmoid** `σ(x) = 1 / (1 + e^{-x})`, these limits are `1` and `0` respectively. The function **saturates** — it gets stuck against its asymptotes.

This is not just trivia. It's the original sin of deep learning:

- When `|x|` is large, `σ(x)` is nearly flat. Its **slope** is close to zero.
- Backprop multiplies the slopes of every layer's activation. If many layers are saturated, you multiply a chain of nearly-zero numbers.
- The gradient at the bottom of the network *vanishes*. The weights stop updating. The network stops learning.

This is the **vanishing gradient problem** — the single biggest reason sigmoids were replaced by ReLU in 2010s deep learning. ReLU doesn't saturate on the positive side (`lim_{x→∞} ReLU(x) = ∞`), so its slope stays at 1 no matter how large `x` gets.

Another flavor of "limit at infinity" worth knowing: **temperature in softmax**. With temperature `T`, softmax becomes `softmax_T(x_i) = exp(x_i / T) / Σ exp(x_j / T)`. The limits:

- `T → 0` — the largest entry dominates; softmax → **argmax** (one-hot).
- `T → ∞` — all entries are nearly equal; softmax → **uniform distribution**.

You'll see this in temperature sampling for language models and in distillation.


In [ ]:
# Sigmoid saturation: function value AND derivative as x grows
def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))

print("x       sigmoid(x)        sigmoid slope (derivative)")
print("-" * 55)
for x in [-10, -5, -2, 0, 2, 5, 10]:
    sig = sigmoid(x)
    slope = sig * (1 - sig)   # the analytical derivative — we'll derive this next notebook
    print(f"{x:>4}    {sig:.10f}      {slope:.10f}")

print("\nNotice: at x = ±10, the slope is essentially zero.")
print("Stacking many such layers multiplies tiny numbers → vanishing gradients.")


## The slope of a curve — a preview of the derivative

Limits give us the language. The very next thing we do with that language is define the **derivative** — the slope of a curve at a single point.

Pick a smooth function `f` and a point `x`. The slope of the line through `(x, f(x))` and `(x + h, f(x + h))` — a **secant line** — is

$$
\frac{f(x + h) - f(x)}{h}
$$

This is the average rate of change over a small interval. Now imagine letting `h` shrink toward zero. The two points slide together. In the limit, the secant becomes the **tangent line** — the line that just kisses the curve at `x`. Its slope is the **instantaneous rate of change**, written

$$
f'(x) = \lim_{h \to 0} \frac{f(x + h) - f(x)}{h}
$$

This *is* the definition of the derivative. Notebook 2 is a careful exploration of what it means, what rules govern it, and what the derivatives of the functions you'll see every day actually are.

For now, the picture: **a derivative is a slope, evaluated at a single point, computed by squeezing a secant line into a tangent**. Hold onto that.


In [ ]:
# Watch the secant slope of f(x) = x^2 at x = 1 converge to the tangent slope (= 2)
f = lambda x: x ** 2
x0 = 1.0
expected_slope = 2 * x0   # f'(x) = 2x analytically

print(f"True tangent slope at x = {x0}: {expected_slope}\n")
print("h          secant slope       error")
print("-" * 45)
for h in [1.0, 0.5, 0.1, 0.01, 0.001, 1e-6]:
    secant = (f(x0 + h) - f(x0)) / h
    print(f"{h:<10g} {secant:.10f}     {abs(secant - expected_slope):.2e}")


## Where this appears in ML

Limits and continuity rarely show up by name in modern ML code, but they're underneath every design decision about activations, losses, and numerical stability:

- **Activation choice.** Continuity guarantees the loss surface has no walls; differentiability guarantees backprop works. ReLU's corner at zero is "good enough" precisely because we almost never sit exactly at zero. GELU, Swish, Mish are continuity-and-smoothness perfectionists.
- **Vanishing gradients.** Saturation = derivative limits to zero at the asymptote. The whole architectural shift from sigmoid/tanh to ReLU (and from RNNs to LSTMs to attention) is one long response to this issue.
- **Exploding gradients.** Limits-at-infinity of activations like `exp(x)` (used inside softmax) blow up. The **log-sum-exp trick** — `log(Σ exp(x_i)) = max(x) + log(Σ exp(x_i - max(x)))` — is a numerical recipe that exploits the *limit* behavior to stay stable.
- **Softmax temperature.** `T → 0` gives argmax; `T → ∞` gives uniform. Temperature sampling in LLMs, distillation losses, contrastive learning — all use this limit explicitly.
- **Cross-entropy and `log(0)`.** Cross-entropy hits `-log(0) = ∞` when the model is fully confident in the wrong answer. Frameworks clamp predictions to `[ε, 1-ε]` for exactly this reason — a numerical version of "the limit doesn't exist, so stop before you reach it."
- **Continuous relaxations of discrete operations.** Argmax is discontinuous; softmax is its continuous relaxation. Gumbel-Softmax, Concrete distribution, and straight-through estimators are all ways to replace discontinuous selections with continuous approximations *so calculus works*.
- **Why we use floats, not booleans, for gradients.** Backprop demands differentiable everything. Any discrete decision in the forward pass needs a continuous surrogate or a special estimator.

Next notebook: **derivatives** — we'll formalize the slope-of-a-curve idea, learn the rules for combining derivatives, and compute the derivative of every activation, loss, and link function ML throws at us.
